# 3.4 Nhãn nhiễu (CIFAR-10, PreAct ResNet-18) trên Kaggle

Chạy 4 cấu hình tốt nhất của Bảng 2 (ERM, ERM + dropout, mixup, mixup + dropout) cho **một mức
nhiễu mỗi phiên** (`NOISE`), rồi gom bảng so với bài báo. Mỗi lần chạy 200 epoch ≈ 1,5–2 giờ GPU,
nên một mức nhiễu (4 lần) vừa một phiên 12 giờ.

**Trước khi chạy** (xem `kaggle/README.md`):
1. Accelerator: **GPU P100** hoặc T4. Internet: **On**.
2. Add Input → dataset chứa thư mục `Source`.
3. Chạy tiếp phiên bị ngắt: Add Input → **output của version cũ**; ô setup tự chép `results/exp34_noise*` về.
4. Chạy thật: đặt `RUN_FULL = True` → **Save Version → Save & Run All (Commit)**.

In [ ]:
# ==== Tham số ====
NOISE = 0.5        # mức nhiễu của phiên này: 0.2 / 0.5 / 0.8
SEEDS = [0]        # thêm 1, 2 nếu còn quota
SMOKE = True       # chạy thử 1 epoch ngắn + kiểm tra resume (vài phút)
RUN_FULL = False   # True: chạy thật 4 cấu hình x len(SEEDS), 200 epoch
AMP = True         # mixed precision (bài báo FP32; ghi rõ trong báo cáo)
WORKERS = 4

In [ ]:
import os, sys, shutil, glob, subprocess, json, time
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CHƯA BẬT GPU")

cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/core/mixup.py', recursive=True)]
assert cands, "Không thấy Source. Add Input -> chọn dataset chứa thư mục Source"
SRC = '/kaggle/working/Source'
if not os.path.exists(SRC):
    shutil.copytree(cands[0], SRC)
os.chdir(SRC)
sys.path.insert(0, SRC)
DATA = '/kaggle/working/data'
OUT = '/kaggle/working/results'

# Chép kết quả dở dang của phiên trước (nếu đã Add Input output cũ) để resume
for ck in glob.glob('/kaggle/input/**/results/exp34_noise*/*/last.pt', recursive=True):
    run_dir = os.path.dirname(ck)
    dst = os.path.join(OUT, os.path.relpath(run_dir, run_dir.split('/results/')[0] + '/results'))
    if not os.path.exists(dst):
        shutil.copytree(run_dir, dst)
        print("khôi phục", dst)

In [ ]:
!python -m pytest tests -q

In [ ]:
def run(args):
    cmd = f"python experiments/exp34_noisy.py --data-dir {DATA} --out-dir {OUT} --workers {WORKERS} "
    cmd += ("--amp " if AMP else "") + args
    print("$", cmd, flush=True)
    t = time.time()
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    print(f"-> xong sau {time.time()-t:.0f}s, exit={rc}", flush=True)
    assert rc == 0, "Lệnh lỗi, xem log phía trên"

## 1. Chạy thử

mixup + dropout, 1 epoch × 20 batch, kiểm tra `summary.json` có lỗi train trên nhãn thật/nhiễu
và tỉ lệ nhãn sai ≈ NOISE·0.9. Sau đó chạy lại với 2 epoch để kiểm tra resume.

In [ ]:
import pandas as pd
if SMOKE:
    base = f"--noise-rate {NOISE} --alpha 8 --dropout 0.3 --limit-batches 20 --run-name smoke34"
    run(base + " --epochs 1 --no-resume")
    s = json.load(open(f"{OUT}/smoke34/summary.json"))
    print({k: s[k] for k in ["train_real_err", "train_corrupted_err", "noise_frac_selected", "noise_frac_wrong"]})
    assert abs(s["noise_frac_wrong"] - NOISE * 0.9) < 0.02, "Tỉ lệ nhãn sai không khớp NOISE·0.9"
    run(base + " --epochs 2")
    h = pd.read_csv(f"{OUT}/smoke34/history.csv")
    assert list(h.epoch) == [0, 1], "Resume sai"
    print("Smoke + resume OK")

## 2. Chạy thật

Lần chạy nào đã có `summary.json` đầy đủ thì bỏ qua; lần nào dở (có `last.pt`) thì tự chạy tiếp.

In [ ]:
from experiments.exp34_report import PAPER
from experiments.exp34_noisy import method_name

configs = [(a, p) for (r, a, p) in PAPER if abs(r - NOISE) < 1e-9]
print("Cấu hình:", configs, "| seeds:", SEEDS)
if RUN_FULL:
    for seed in SEEDS:
        for alpha, p in configs:
            name = f"exp34_noise{round(NOISE * 100)}/{method_name(alpha, p)}_seed{seed}"
            sp = f"{OUT}/{name}/summary.json"
            if os.path.exists(sp) and "train_real_err" in json.load(open(sp)):
                print("đã xong, bỏ qua:", name)
                continue
            run(f"--noise-rate {NOISE} --alpha {alpha:g} --dropout {p:g} --seed {seed} --epochs 200")
else:
    print("Bỏ qua chạy thật (RUN_FULL = False)")

## 3. Bảng so với bài báo + biểu đồ lỗi test theo epoch

In [ ]:
from IPython.display import Markdown, Image, display
!python experiments/exp34_report.py --results {OUT}
if os.path.exists(f"{OUT}/exp34_report.md"):
    display(Markdown(open(f"{OUT}/exp34_report.md", encoding="utf-8").read()))
if os.path.exists(f"{OUT}/exp34_curves.png"):
    display(Image(f"{OUT}/exp34_curves.png"))

In [ ]:
# Đóng gói log (không kèm .pt) để tải về; last.pt vẫn nằm trong Output để resume phiên sau
!cd /kaggle/working && zip -qr results_logs.zip results -x '*.pt' && ls -lh results_logs.zip